# iTransformer Context Forecasting


In [ ]:
MODEL_DIR_NAME = 'iTransform'
MODEL_LABEL = 'iTransformer'
MODEL_TAG = 'itransformer'

from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message=r".*'H' is deprecated.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=r".*'T' is deprecated.*", category=FutureWarning)

# Manual window length settings used by the final baseline experiments.
#
# Frequency | 24h target | 7d target | 28d target | 2d input | 7d input
# 30min     | 48         | 336       | 1344       | 96       | 336
# 1H        | 24         | 168       | 672        | 48       | 168
# 2H        | 12         | 84        | 336        | 24       | 84
# 3H        | 8          | 56        | 224        | 16       | 56
FREQ = "30min"
PRIMARY_HORIZON = "24h"
SEQ_LEN = 48
INPUT_LEN = 96
TRAIN_STRIDE = SEQ_LEN

SEED = 0
VAL_FRAC = 0.20
BATCH_SIZE = 64
MAX_STEPS = 300
EVAL_WINDOWS = 512
MAX_TRAIN_WINDOWS_PER_CUSTOMER = 64
NUM_LOADER_WORKERS = 2 if sys.platform != "win32" else 0
QUANTILES = [0.1, 0.5, 0.9]
ALPHA_PI = 0.20
BINS = 80

HOUSEHOLD_ID_COL = "CUSTOMER_KEY"
TIME_COL = "READING_DATETIME"
TARGET_COL = "kWh"

STATIC_BASE_COLS = [
    "NUM_OCCUPANTS", "NUM_ROOMS_HEATED", "NUM_REFRIGERATORS",
    "Unit", "SemiDetached", "SeparateHouse",
    "HAS_GAS_HEATING", "HAS_GAS_HOT_WATER", "HAS_GAS_COOKING",
    "HAS_POOLPUMP", "Ducted", "SplitSystem", "NoAirCon", "OtherAirCon",
    "CONTROLLED_LOAD_CNT",
]

STATIC_CAT_CANDIDATES = [
    "NEAREST_STATION_NO", "STATION_NO", "station_id", "StationID",
    "TrialRegion", "TRIAL_REGION", "TrialRegionID", "trial_region_id",
]

PAST_COV_COLS = [
    "HourSin", "HourCos", "HalfHourSin", "HalfHourCos",
    "WeekdaySin", "WeekdayCos", "MonthSin", "MonthCos",
    "Summer", "Fall", "Winter", "Spring",
    "Temperature", "CDD", "HDD", "wind_speed",
    "Temperature_lag_2", "Temperature_lag_6",
    "Temperature_lag_48", "Temperature_lag_144",
    "temperature_lag_2", "temperature_lag_6",
    "temperature_lag_48", "temperature_lag_144",
]

FUTURE_COV_COLS = [
    "HourSin", "HourCos", "HalfHourSin", "HalfHourCos",
    "WeekdaySin", "WeekdayCos", "MonthSin", "MonthCos",
    "Summer", "Fall", "Winter", "Spring",
]

def pandas_freq(freq):
    return freq.replace("H", "h")


# All cleaned notebooks use one fixed dataset location.
SORTED_DIR = Path("..")
DATA_PATH = SORTED_DIR / "data_with_weather.pickle"
MODEL_DIR = SORTED_DIR / MODEL_DIR_NAME
print(f"MODEL_DIR={MODEL_DIR}")
print(
    f"FREQ={FREQ} | horizon={PRIMARY_HORIZON} | SEQ_LEN={SEQ_LEN} | "
    f"INPUT_LEN={INPUT_LEN} | TRAIN_STRIDE={TRAIN_STRIDE}"
)


### Preprocessing Roadmap

The preprocessing stage is intentionally a short pipeline rather than a model-specific trick:

1. Load the shared household energy table.
2. Keep the columns needed for this experiment and fill any missing optional covariates.
3. Convert timestamps, customer IDs, numeric covariates, and categorical IDs into model-ready types.
4. Add cyclical calendar features so hour, weekday, and month wrap around naturally.
5. Resample each customer to the selected frequency.
6. Split by customer so validation households are unseen during training.
7. Build context/target windows or library time-series objects, then scale the model inputs.

The small checks in this section are kept only where they prevent silent data leakage, missing-column errors, or invalid tensor shapes.


In [ ]:
# Place the dataset in the sorted directory, one level above this model notebook.
df = pd.read_pickle(DATA_PATH).copy()
print("Data file:", DATA_PATH)

required_columns = [TIME_COL, TARGET_COL, HOUSEHOLD_ID_COL]
missing_required = [column for column in required_columns if column not in df.columns]
if missing_required:
    raise KeyError(f"Dataset is missing required columns: {missing_required}")

# Cyclical calendar values are known for both the historical and forecast windows.
def add_calendar_covariates(frame, time_values):
    dt = pd.DatetimeIndex(pd.to_datetime(time_values))
    hour = dt.hour.astype(np.float32)
    minute = dt.minute.astype(np.float32)
    half_hour = hour * 2 + (minute // 30)
    weekday = dt.weekday.astype(np.float32)
    month = dt.month.astype(np.float32)

    frame["HourSin"] = np.sin(2.0 * np.pi * hour / 24.0).astype("float32")
    frame["HourCos"] = np.cos(2.0 * np.pi * hour / 24.0).astype("float32")
    frame["HalfHourSin"] = np.sin(2.0 * np.pi * half_hour / 48.0).astype("float32")
    frame["HalfHourCos"] = np.cos(2.0 * np.pi * half_hour / 48.0).astype("float32")
    frame["WeekdaySin"] = np.sin(2.0 * np.pi * weekday / 7.0).astype("float32")
    frame["WeekdayCos"] = np.cos(2.0 * np.pi * weekday / 7.0).astype("float32")
    frame["MonthSin"] = np.sin(2.0 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["MonthCos"] = np.cos(2.0 * np.pi * (month - 1.0) / 12.0).astype("float32")
    frame["Summer"] = np.isin(month, [12, 1, 2]).astype("float32")
    frame["Fall"] = np.isin(month, [3, 4, 5]).astype("float32")
    frame["Winter"] = np.isin(month, [6, 7, 8]).astype("float32")
    frame["Spring"] = np.isin(month, [9, 10, 11]).astype("float32")
    return frame


def static_cat_code_col(column):
    return "cat_" + re.sub(r"[^A-Za-z0-9]+", "_", column).strip("_")


# Keep the feature layout stable when an optional numerical field is absent.
if "NUM_OCCUPANTS" not in df.columns: df["NUM_OCCUPANTS"] = 0.0
if "NUM_ROOMS_HEATED" not in df.columns: df["NUM_ROOMS_HEATED"] = 0.0
if "NUM_REFRIGERATORS" not in df.columns: df["NUM_REFRIGERATORS"] = 0.0
if "Unit" not in df.columns: df["Unit"] = 0.0
if "SemiDetached" not in df.columns: df["SemiDetached"] = 0.0
if "SeparateHouse" not in df.columns: df["SeparateHouse"] = 0.0
if "HAS_GAS_HEATING" not in df.columns: df["HAS_GAS_HEATING"] = 0.0
if "HAS_GAS_HOT_WATER" not in df.columns: df["HAS_GAS_HOT_WATER"] = 0.0
if "HAS_GAS_COOKING" not in df.columns: df["HAS_GAS_COOKING"] = 0.0
if "HAS_POOLPUMP" not in df.columns: df["HAS_POOLPUMP"] = 0.0
if "Ducted" not in df.columns: df["Ducted"] = 0.0
if "SplitSystem" not in df.columns: df["SplitSystem"] = 0.0
if "NoAirCon" not in df.columns: df["NoAirCon"] = 0.0
if "OtherAirCon" not in df.columns: df["OtherAirCon"] = 0.0
if "CONTROLLED_LOAD_CNT" not in df.columns: df["CONTROLLED_LOAD_CNT"] = 0.0
if "Temperature" not in df.columns: df["Temperature"] = 0.0
temperature_for_degree_days = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0)
if "CDD" not in df.columns: df["CDD"] = np.maximum(temperature_for_degree_days - 18.0, 0.0).astype("float32")
if "HDD" not in df.columns: df["HDD"] = np.maximum(18.0 - temperature_for_degree_days, 0.0).astype("float32")
if "wind_speed" not in df.columns: df["wind_speed"] = 0.0
if "Temperature_lag_2" not in df.columns: df["Temperature_lag_2"] = 0.0
if "Temperature_lag_6" not in df.columns: df["Temperature_lag_6"] = 0.0
if "Temperature_lag_48" not in df.columns: df["Temperature_lag_48"] = 0.0
if "Temperature_lag_144" not in df.columns: df["Temperature_lag_144"] = 0.0
if "temperature_lag_2" not in df.columns: df["temperature_lag_2"] = 0.0
if "temperature_lag_6" not in df.columns: df["temperature_lag_6"] = 0.0
if "temperature_lag_48" not in df.columns: df["temperature_lag_48"] = 0.0
if "temperature_lag_144" not in df.columns: df["temperature_lag_144"] = 0.0

# Convert identifiers and timestamps before sorting or resampling.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df[HOUSEHOLD_ID_COL] = pd.to_numeric(df[HOUSEHOLD_ID_COL], errors="coerce").astype("int64")
df = df.sort_values([HOUSEHOLD_ID_COL, TIME_COL])
df = add_calendar_covariates(df, df[TIME_COL])

# Convert model inputs explicitly so the preprocessing can be read in execution order.
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0.0).astype("float32")
df["NUM_OCCUPANTS"] = pd.to_numeric(df["NUM_OCCUPANTS"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_ROOMS_HEATED"] = pd.to_numeric(df["NUM_ROOMS_HEATED"], errors="coerce").fillna(0.0).astype("float32")
df["NUM_REFRIGERATORS"] = pd.to_numeric(df["NUM_REFRIGERATORS"], errors="coerce").fillna(0.0).astype("float32")
df["Unit"] = pd.to_numeric(df["Unit"], errors="coerce").fillna(0.0).astype("float32")
df["SemiDetached"] = pd.to_numeric(df["SemiDetached"], errors="coerce").fillna(0.0).astype("float32")
df["SeparateHouse"] = pd.to_numeric(df["SeparateHouse"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HEATING"] = pd.to_numeric(df["HAS_GAS_HEATING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_HOT_WATER"] = pd.to_numeric(df["HAS_GAS_HOT_WATER"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_GAS_COOKING"] = pd.to_numeric(df["HAS_GAS_COOKING"], errors="coerce").fillna(0.0).astype("float32")
df["HAS_POOLPUMP"] = pd.to_numeric(df["HAS_POOLPUMP"], errors="coerce").fillna(0.0).astype("float32")
df["Ducted"] = pd.to_numeric(df["Ducted"], errors="coerce").fillna(0.0).astype("float32")
df["SplitSystem"] = pd.to_numeric(df["SplitSystem"], errors="coerce").fillna(0.0).astype("float32")
df["NoAirCon"] = pd.to_numeric(df["NoAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["OtherAirCon"] = pd.to_numeric(df["OtherAirCon"], errors="coerce").fillna(0.0).astype("float32")
df["CONTROLLED_LOAD_CNT"] = pd.to_numeric(df["CONTROLLED_LOAD_CNT"], errors="coerce").fillna(0.0).astype("float32")
df["HourSin"] = pd.to_numeric(df["HourSin"], errors="coerce").fillna(0.0).astype("float32")
df["HourCos"] = pd.to_numeric(df["HourCos"], errors="coerce").fillna(0.0).astype("float32")
df["HalfHourSin"] = pd.to_numeric(df["HalfHourSin"], errors="coerce").fillna(0.0).astype("float32")
df["HalfHourCos"] = pd.to_numeric(df["HalfHourCos"], errors="coerce").fillna(0.0).astype("float32")
df["WeekdaySin"] = pd.to_numeric(df["WeekdaySin"], errors="coerce").fillna(0.0).astype("float32")
df["WeekdayCos"] = pd.to_numeric(df["WeekdayCos"], errors="coerce").fillna(0.0).astype("float32")
df["MonthSin"] = pd.to_numeric(df["MonthSin"], errors="coerce").fillna(0.0).astype("float32")
df["MonthCos"] = pd.to_numeric(df["MonthCos"], errors="coerce").fillna(0.0).astype("float32")
df["Summer"] = pd.to_numeric(df["Summer"], errors="coerce").fillna(0.0).astype("float32")
df["Fall"] = pd.to_numeric(df["Fall"], errors="coerce").fillna(0.0).astype("float32")
df["Winter"] = pd.to_numeric(df["Winter"], errors="coerce").fillna(0.0).astype("float32")
df["Spring"] = pd.to_numeric(df["Spring"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature"] = pd.to_numeric(df["Temperature"], errors="coerce").fillna(0.0).astype("float32")
df["CDD"] = pd.to_numeric(df["CDD"], errors="coerce").fillna(0.0).astype("float32")
df["HDD"] = pd.to_numeric(df["HDD"], errors="coerce").fillna(0.0).astype("float32")
df["wind_speed"] = pd.to_numeric(df["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_2"] = pd.to_numeric(df["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_6"] = pd.to_numeric(df["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_48"] = pd.to_numeric(df["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["Temperature_lag_144"] = pd.to_numeric(df["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_2"] = pd.to_numeric(df["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_6"] = pd.to_numeric(df["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_48"] = pd.to_numeric(df["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
df["temperature_lag_144"] = pd.to_numeric(df["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")

# Preserve the categorical feature handling used by the trained baselines.
STATIC_CAT_CODE_COLS = []
if "NEAREST_STATION_NO" in df.columns:
    code_col = static_cat_code_col("NEAREST_STATION_NO")
    codes, _ = pd.factorize(df["NEAREST_STATION_NO"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "STATION_NO" in df.columns:
    code_col = static_cat_code_col("STATION_NO")
    codes, _ = pd.factorize(df["STATION_NO"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "station_id" in df.columns:
    code_col = static_cat_code_col("station_id")
    codes, _ = pd.factorize(df["station_id"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "StationID" in df.columns:
    code_col = static_cat_code_col("StationID")
    codes, _ = pd.factorize(df["StationID"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "TrialRegion" in df.columns:
    code_col = static_cat_code_col("TrialRegion")
    codes, _ = pd.factorize(df["TrialRegion"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "TRIAL_REGION" in df.columns:
    code_col = static_cat_code_col("TRIAL_REGION")
    codes, _ = pd.factorize(df["TRIAL_REGION"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "TrialRegionID" in df.columns:
    code_col = static_cat_code_col("TrialRegionID")
    codes, _ = pd.factorize(df["TrialRegionID"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
if "trial_region_id" in df.columns:
    code_col = static_cat_code_col("trial_region_id")
    codes, _ = pd.factorize(df["trial_region_id"].astype("string").fillna("__missing__"), sort=True)
    df[code_col] = codes.astype("float32")
    STATIC_CAT_CODE_COLS.append(code_col)
STATIC_COLS = STATIC_BASE_COLS + STATIC_CAT_CODE_COLS

# Sum interval energy, average time-varying fields, and preserve household attributes.
aggregation = {
    TARGET_COL: "sum",
    "HourSin": "mean",
    "HourCos": "mean",
    "HalfHourSin": "mean",
    "HalfHourCos": "mean",
    "WeekdaySin": "mean",
    "WeekdayCos": "mean",
    "MonthSin": "mean",
    "MonthCos": "mean",
    "Summer": "mean",
    "Fall": "mean",
    "Winter": "mean",
    "Spring": "mean",
    "Temperature": "mean",
    "CDD": "mean",
    "HDD": "mean",
    "wind_speed": "mean",
    "Temperature_lag_2": "mean",
    "Temperature_lag_6": "mean",
    "Temperature_lag_48": "mean",
    "Temperature_lag_144": "mean",
    "temperature_lag_2": "mean",
    "temperature_lag_6": "mean",
    "temperature_lag_48": "mean",
    "temperature_lag_144": "mean",
    "NUM_OCCUPANTS": "first",
    "NUM_ROOMS_HEATED": "first",
    "NUM_REFRIGERATORS": "first",
    "Unit": "first",
    "SemiDetached": "first",
    "SeparateHouse": "first",
    "HAS_GAS_HEATING": "first",
    "HAS_GAS_HOT_WATER": "first",
    "HAS_GAS_COOKING": "first",
    "HAS_POOLPUMP": "first",
    "Ducted": "first",
    "SplitSystem": "first",
    "NoAirCon": "first",
    "OtherAirCon": "first",
    "CONTROLLED_LOAD_CNT": "first",
}
for code_col in STATIC_CAT_CODE_COLS:
    aggregation[code_col] = "first"

df = (
    df.set_index(TIME_COL)
      .groupby(HOUSEHOLD_ID_COL)
      .resample(pandas_freq(FREQ))
      .agg(aggregation)
      .reset_index()
      .sort_values([HOUSEHOLD_ID_COL, TIME_COL])
      .reset_index(drop=True)
)

# Calendar values are recalculated from the final resampled timestamps.
df = add_calendar_covariates(df, df[TIME_COL])

# Split complete households so no customer's windows appear in both partitions.
rng = np.random.default_rng(SEED)
all_customers = np.array(sorted(df[HOUSEHOLD_ID_COL].dropna().unique()))
n_val = max(1, int(len(all_customers) * VAL_FRAC))
val_customers = set(rng.choice(all_customers, size=n_val, replace=False).tolist())
train_customers = set(all_customers.tolist()) - val_customers

print("Preprocessed dataframe shape:", df.shape)
print(
    f"Customers total={len(all_customers)} | "
    f"train={len(train_customers)} | val={len(val_customers)}"
)
print("Past covariates:", len(PAST_COV_COLS), "| Future covariates:", len(FUTURE_COV_COLS), "| Static features:", len(STATIC_COLS))


In [ ]:
# These seven metrics form the complete evaluation set used by the cleaned notebooks.
METRIC_NAMES = [
    "MAE",
    "RMSE",
    "PeakMAE",
    "QuantileLoss",
    "KL_Divergence",
    "DTW",
    "WinklerScore",
]


def pinball_loss(y_true, prediction, quantile):
    error = np.asarray(y_true, dtype=np.float64) - np.asarray(
        prediction,
        dtype=np.float64,
    )
    return float(
        np.nanmean(
            np.maximum(
                quantile * error,
                (quantile - 1.0) * error,
            )
        )
    )


def finite_flat(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    return values[np.isfinite(values)]


def kl_divergence(real, forecast):
    real_flat = finite_flat(real)
    forecast_flat = finite_flat(forecast)
    lower = float(real_flat.min())
    upper = float(real_flat.max())
    if upper <= lower:
        upper = lower + 1e-6

    edges = np.linspace(lower, upper, BINS + 1)
    real_histogram, _ = np.histogram(real_flat, bins=edges)
    forecast_histogram, _ = np.histogram(forecast_flat, bins=edges)

    epsilon = 1e-8
    p = real_histogram.astype(np.float64) + epsilon
    q = forecast_histogram.astype(np.float64) + epsilon
    p /= p.sum()
    q /= q.sum()
    return float(np.sum(p * np.log(p / q)))


def dtw_distance(real, forecast, max_points=256):
    real = np.asarray(real, dtype=np.float64).reshape(-1)
    forecast = np.asarray(forecast, dtype=np.float64).reshape(-1)

    if len(real) > max_points:
        indices = np.linspace(0, len(real) - 1, max_points).round().astype(int)
        real = real[indices]
        forecast = forecast[indices]

    previous = np.full(len(forecast) + 1, np.inf)
    current = np.full(len(forecast) + 1, np.inf)
    previous[0] = 0.0

    for row in range(1, len(real) + 1):
        current[0] = np.inf
        for column in range(1, len(forecast) + 1):
            difference = abs(real[row - 1] - forecast[column - 1])
            current[column] = difference + min(
                previous[column],
                current[column - 1],
                previous[column - 1],
            )
        previous, current = current, previous
    return float(previous[-1])


def compute_metrics(y_real, lower, median, upper):
    y_real = np.asarray(y_real, dtype=np.float64)
    lower = np.asarray(lower, dtype=np.float64)
    median = np.asarray(median, dtype=np.float64)
    upper = np.asarray(upper, dtype=np.float64)

    interval_lower = np.minimum(lower, upper)
    interval_upper = np.maximum(lower, upper)

    MAE = float(np.nanmean(np.abs(median - y_real)))
    RMSE = float(np.sqrt(np.nanmean((median - y_real) ** 2)))
    PeakMAE = float(
        np.nanmean(
            np.abs(
                np.nanmax(median[:, :, 0], axis=1)
                - np.nanmax(y_real[:, :, 0], axis=1)
            )
        )
    )

    quantile_losses = [
        pinball_loss(y_real, interval_lower, 0.1),
        pinball_loss(y_real, median, 0.5),
        pinball_loss(y_real, interval_upper, 0.9),
    ]
    QuantileLoss = float(np.mean(quantile_losses))
    KL_Divergence = kl_divergence(y_real, median)

    dtw_count = min(32, y_real.shape[0])
    DTW = float(
        np.nanmean(
            [
                dtw_distance(y_real[index, :, 0], median[index, :, 0])
                for index in range(dtw_count)
            ]
        )
    )

    interval_width = interval_upper - interval_lower
    below = y_real < interval_lower
    above = y_real > interval_upper
    WinklerScore = float(
        np.nanmean(
            interval_width
            + (2.0 / ALPHA_PI) * (interval_lower - y_real) * below
            + (2.0 / ALPHA_PI) * (y_real - interval_upper) * above
        )
    )

    return {
        "MAE": MAE,
        "RMSE": RMSE,
        "PeakMAE": PeakMAE,
        "QuantileLoss": QuantileLoss,
        "KL_Divergence": KL_Divergence,
        "DTW": DTW,
        "WinklerScore": WinklerScore,
    }


def print_metrics_table(rows):
    table = pd.DataFrame(rows)
    table = table[METRIC_NAMES]
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(table.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
    return table


In [ ]:
def dataloader_kwargs():
    kwargs = {"num_workers": NUM_LOADER_WORKERS}
    if NUM_LOADER_WORKERS > 0:
        kwargs["persistent_workers"] = True
    if torch.cuda.is_available():
        kwargs["pin_memory"] = True
    return kwargs


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("medium")

past_cov_cols = list(PAST_COV_COLS)
future_cov_cols = list(FUTURE_COV_COLS)
static_cols = list(STATIC_COLS)
window_rng = np.random.default_rng(SEED)

y_past_train_parts = []
y_future_train_parts = []
past_cov_train_parts = []
future_cov_train_parts = []
static_train_parts = []

y_past_val_parts = []
y_future_val_parts = []
past_cov_val_parts = []
future_cov_val_parts = []
static_val_parts = []

# Build fixed windows one customer at a time, matching the generative notebook layout.
for household_id, customer in df.groupby(HOUSEHOLD_ID_COL):
    customer = customer.sort_values(TIME_COL).drop_duplicates(TIME_COL, keep="last")
    if customer.empty:
        continue

    complete_index = pd.date_range(
        customer[TIME_COL].min(),
        customer[TIME_COL].max(),
        freq=pandas_freq(FREQ),
    )
    customer = customer.set_index(TIME_COL).reindex(complete_index)
    customer.index.name = TIME_COL
    customer[HOUSEHOLD_ID_COL] = household_id

    # Static values are propagated over any missing timestamps.
    customer["NUM_OCCUPANTS"] = pd.to_numeric(customer["NUM_OCCUPANTS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["NUM_ROOMS_HEATED"] = pd.to_numeric(customer["NUM_ROOMS_HEATED"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["NUM_REFRIGERATORS"] = pd.to_numeric(customer["NUM_REFRIGERATORS"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["Unit"] = pd.to_numeric(customer["Unit"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["SemiDetached"] = pd.to_numeric(customer["SemiDetached"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["SeparateHouse"] = pd.to_numeric(customer["SeparateHouse"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_GAS_HEATING"] = pd.to_numeric(customer["HAS_GAS_HEATING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_GAS_HOT_WATER"] = pd.to_numeric(customer["HAS_GAS_HOT_WATER"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_GAS_COOKING"] = pd.to_numeric(customer["HAS_GAS_COOKING"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["HAS_POOLPUMP"] = pd.to_numeric(customer["HAS_POOLPUMP"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["Ducted"] = pd.to_numeric(customer["Ducted"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["SplitSystem"] = pd.to_numeric(customer["SplitSystem"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["NoAirCon"] = pd.to_numeric(customer["NoAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["OtherAirCon"] = pd.to_numeric(customer["OtherAirCon"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    customer["CONTROLLED_LOAD_CNT"] = pd.to_numeric(customer["CONTROLLED_LOAD_CNT"], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")
    for code_col in STATIC_CAT_CODE_COLS:
        customer[code_col] = pd.to_numeric(customer[code_col], errors="coerce").ffill().bfill().fillna(0.0).astype("float32")

    # Recreate calendar values on the complete index and fill other temporal gaps.
    customer = add_calendar_covariates(customer, customer.index)
    customer[TARGET_COL] = pd.to_numeric(customer[TARGET_COL], errors="coerce").fillna(0.0).astype("float32")
    customer["HourSin"] = pd.to_numeric(customer["HourSin"], errors="coerce").fillna(0.0).astype("float32")
    customer["HourCos"] = pd.to_numeric(customer["HourCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["HalfHourSin"] = pd.to_numeric(customer["HalfHourSin"], errors="coerce").fillna(0.0).astype("float32")
    customer["HalfHourCos"] = pd.to_numeric(customer["HalfHourCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["WeekdaySin"] = pd.to_numeric(customer["WeekdaySin"], errors="coerce").fillna(0.0).astype("float32")
    customer["WeekdayCos"] = pd.to_numeric(customer["WeekdayCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["MonthSin"] = pd.to_numeric(customer["MonthSin"], errors="coerce").fillna(0.0).astype("float32")
    customer["MonthCos"] = pd.to_numeric(customer["MonthCos"], errors="coerce").fillna(0.0).astype("float32")
    customer["Summer"] = pd.to_numeric(customer["Summer"], errors="coerce").fillna(0.0).astype("float32")
    customer["Fall"] = pd.to_numeric(customer["Fall"], errors="coerce").fillna(0.0).astype("float32")
    customer["Winter"] = pd.to_numeric(customer["Winter"], errors="coerce").fillna(0.0).astype("float32")
    customer["Spring"] = pd.to_numeric(customer["Spring"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature"] = pd.to_numeric(customer["Temperature"], errors="coerce").fillna(0.0).astype("float32")
    customer["CDD"] = pd.to_numeric(customer["CDD"], errors="coerce").fillna(0.0).astype("float32")
    customer["HDD"] = pd.to_numeric(customer["HDD"], errors="coerce").fillna(0.0).astype("float32")
    customer["wind_speed"] = pd.to_numeric(customer["wind_speed"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_2"] = pd.to_numeric(customer["Temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_6"] = pd.to_numeric(customer["Temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_48"] = pd.to_numeric(customer["Temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    customer["Temperature_lag_144"] = pd.to_numeric(customer["Temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_2"] = pd.to_numeric(customer["temperature_lag_2"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_6"] = pd.to_numeric(customer["temperature_lag_6"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_48"] = pd.to_numeric(customer["temperature_lag_48"], errors="coerce").fillna(0.0).astype("float32")
    customer["temperature_lag_144"] = pd.to_numeric(customer["temperature_lag_144"], errors="coerce").fillna(0.0).astype("float32")

    if len(customer) < INPUT_LEN + SEQ_LEN:
        continue

    y_all = customer[[TARGET_COL]].to_numpy(dtype=np.float32)
    past_cov_all = customer[past_cov_cols].to_numpy(dtype=np.float32)
    future_cov_all = customer[future_cov_cols].to_numpy(dtype=np.float32)
    static_values = customer[static_cols].iloc[0].to_numpy(dtype=np.float32)

    starts = np.arange(
        0,
        len(customer) - INPUT_LEN - SEQ_LEN + 1,
        TRAIN_STRIDE,
    )
    is_validation = household_id in val_customers
    if not is_validation and len(starts) > MAX_TRAIN_WINDOWS_PER_CUSTOMER:
        starts = np.sort(
            window_rng.choice(
                starts,
                size=MAX_TRAIN_WINDOWS_PER_CUSTOMER,
                replace=False,
            )
        )

    for start in starts:
        middle = int(start + INPUT_LEN)
        end = int(middle + SEQ_LEN)

        if is_validation:
            y_past_val_parts.append(y_all[start:middle])
            y_future_val_parts.append(y_all[middle:end])
            past_cov_val_parts.append(past_cov_all[start:middle])
            future_cov_val_parts.append(future_cov_all[middle:end])
            static_val_parts.append(static_values)
        else:
            y_past_train_parts.append(y_all[start:middle])
            y_future_train_parts.append(y_all[middle:end])
            past_cov_train_parts.append(past_cov_all[start:middle])
            future_cov_train_parts.append(future_cov_all[middle:end])
            static_train_parts.append(static_values)

if not y_future_train_parts or not y_future_val_parts:
    raise RuntimeError("The customer split did not produce usable train and validation windows.")

# Stack each input explicitly so its final three-dimensional shape is visible.
y_past_train = np.stack(y_past_train_parts).astype(np.float32)
y_future_train = np.stack(y_future_train_parts).astype(np.float32)
past_cov_train = np.stack(past_cov_train_parts).astype(np.float32)
future_cov_train = np.stack(future_cov_train_parts).astype(np.float32)
static_train = np.stack(static_train_parts).astype(np.float32)

y_past_val = np.stack(y_past_val_parts).astype(np.float32)
y_future_val = np.stack(y_future_val_parts).astype(np.float32)
past_cov_val = np.stack(past_cov_val_parts).astype(np.float32)
future_cov_val = np.stack(future_cov_val_parts).astype(np.float32)
static_val = np.stack(static_val_parts).astype(np.float32)

pools = {
    "train": {
        "yp": y_past_train,
        "yf": y_future_train,
        "pc": past_cov_train,
        "fc": future_cov_train,
        "xs": static_train,
    },
    "validation": {
        "yp": y_past_val,
        "yf": y_future_val,
        "pc": past_cov_val,
        "fc": future_cov_val,
        "xs": static_val,
    },
}

print("Train y_past:", y_past_train.shape, "| y_future:", y_future_train.shape)
print("Validation y_past:", y_past_val.shape, "| y_future:", y_future_val.shape)
print("Past covariates:", past_cov_cols)
print("Future covariates:", future_cov_cols)
print("Static covariates:", static_cols)

# Fit target and covariate scalers on training windows only.
y_scaler = StandardScaler()
y_train_log = np.concatenate([
    np.log1p(y_past_train).reshape(-1, 1),
    np.log1p(y_future_train).reshape(-1, 1),
])
y_scaler.fit(y_train_log)

pc_scaler = StandardScaler()
fc_scaler = StandardScaler()
xs_scaler = StandardScaler()
pc_scaler.fit(past_cov_train.reshape(-1, past_cov_train.shape[-1]))
fc_scaler.fit(future_cov_train.reshape(-1, future_cov_train.shape[-1]))
xs_scaler.fit(static_train)

scaled_train = {
    "yp": y_scaler.transform(np.log1p(y_past_train).reshape(-1, 1)).reshape(y_past_train.shape).astype(np.float32),
    "yf": y_scaler.transform(np.log1p(y_future_train).reshape(-1, 1)).reshape(y_future_train.shape).astype(np.float32),
    "pc": pc_scaler.transform(past_cov_train.reshape(-1, past_cov_train.shape[-1])).reshape(past_cov_train.shape).astype(np.float32),
    "fc": fc_scaler.transform(future_cov_train.reshape(-1, future_cov_train.shape[-1])).reshape(future_cov_train.shape).astype(np.float32),
    "xs": xs_scaler.transform(static_train).astype(np.float32),
}

scaled_val = {
    "yp": y_scaler.transform(np.log1p(y_past_val).reshape(-1, 1)).reshape(y_past_val.shape).astype(np.float32),
    "yf": y_scaler.transform(np.log1p(y_future_val).reshape(-1, 1)).reshape(y_future_val.shape).astype(np.float32),
    "pc": pc_scaler.transform(past_cov_val.reshape(-1, past_cov_val.shape[-1])).reshape(past_cov_val.shape).astype(np.float32),
    "fc": fc_scaler.transform(future_cov_val.reshape(-1, future_cov_val.shape[-1])).reshape(future_cov_val.shape).astype(np.float32),
    "xs": xs_scaler.transform(static_val).astype(np.float32),
}


class ForecastWindowDataset(Dataset):
    def __init__(self, pool):
        self.pool = pool

    def __len__(self):
        return self.pool["yf"].shape[0]

    def __getitem__(self, index):
        return (
            torch.from_numpy(self.pool["yp"][index]),
            torch.from_numpy(self.pool["yf"][index]),
            torch.from_numpy(self.pool["pc"][index]),
            torch.from_numpy(self.pool["fc"][index]),
            torch.from_numpy(self.pool["xs"][index]),
        )


loader = DataLoader(
    ForecastWindowDataset(scaled_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    **dataloader_kwargs(),
)
print("Training windows:", len(loader.dataset), "| Batches:", len(loader))


In [ ]:
HIDDEN_SIZE = 128
TRANSFORMER_LAYERS = 3
TRANSFORMER_HEADS = 4
TRANSFORMER_FF = 512
DROPOUT = 0.10
LR = 2e-4

# Taken from the iTransformer official implementation: https://github.com/thuml/iTransformer
class LocalITransformer(nn.Module):
    """Treat each variable history as one Transformer token."""

    def __init__(self, past_cov_dim, future_cov_dim, static_dim):
        super().__init__()
        self.future_cov_dim = future_cov_dim
        self.static_dim = static_dim

        self.variable_projection = nn.Linear(INPUT_LEN, HIDDEN_SIZE)
        self.future_projection = nn.Linear(max(1, SEQ_LEN * future_cov_dim), HIDDEN_SIZE)
        self.static_projection = nn.Linear(max(1, static_dim), HIDDEN_SIZE)

        layer = nn.TransformerEncoderLayer(
            d_model=HIDDEN_SIZE,
            nhead=TRANSFORMER_HEADS,
            dim_feedforward=TRANSFORMER_FF,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=TRANSFORMER_LAYERS,
        )
        self.norm = nn.LayerNorm(HIDDEN_SIZE)
        self.forecast_head = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_SIZE, SEQ_LEN * 3),
        )

    def forward(self, y_past, past_covariates, future_covariates, static):
        # (B, input length, variables) -> (B, variables, input length)
        variable_history = torch.cat(
            [y_past, past_covariates], dim=-1
        ).transpose(1, 2)
        variable_tokens = self.variable_projection(variable_history)

        if self.future_cov_dim > 0:
            future_input = future_covariates.flatten(1)
        else:
            future_input = future_covariates.new_zeros(future_covariates.shape[0], 1)
        future_token = self.future_projection(future_input).unsqueeze(1)

        if self.static_dim > 0:
            static_input = static
        else:
            static_input = y_past.new_zeros(y_past.shape[0], 1)
        static_token = self.static_projection(static_input).unsqueeze(1)

        tokens = torch.cat(
            [variable_tokens, future_token, static_token],
            dim=1,
        )
        encoded = self.encoder(tokens)
        pooled = self.norm(encoded.mean(dim=1))
        raw = self.forecast_head(pooled).view(-1, SEQ_LEN, 3)

        median = raw[:, :, 1:2]
        lower = median - F.softplus(raw[:, :, 0:1])
        upper = median + F.softplus(raw[:, :, 2:3])
        return torch.cat([lower, median, upper], dim=-1)


def torch_pinball(y, prediction, quantile):
    error = y - prediction
    return torch.maximum(
        quantile * error,
        (quantile - 1.0) * error,
    ).mean()


model = LocalITransformer(
    past_cov_dim=scaled_train["pc"].shape[-1],
    future_cov_dim=scaled_train["fc"].shape[-1],
    static_dim=scaled_train["xs"].shape[-1],
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)


In [ ]:
# Train a new model. Run this cell before the save cell.
step = 0
while step < MAX_STEPS:
    model.train()
    for y_past_b, y_future_b, pc_b, fc_b, xs_b in loader:
        y_past_b = y_past_b.to(DEVICE, non_blocking=True)
        y_future_b = y_future_b.to(DEVICE, non_blocking=True)
        pc_b = pc_b.to(DEVICE, non_blocking=True)
        fc_b = fc_b.to(DEVICE, non_blocking=True)
        xs_b = xs_b.to(DEVICE, non_blocking=True)

        quantiles = model(y_past_b, pc_b, fc_b, xs_b)
        loss = (
            torch_pinball(y_future_b, quantiles[:, :, 0:1], 0.1)
            + torch_pinball(y_future_b, quantiles[:, :, 1:2], 0.5)
            + torch_pinball(y_future_b, quantiles[:, :, 2:3], 0.9)
        ) / 3.0

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if step % 20 == 0:
            print(
                f"[{MODEL_LABEL}] step {step:04d}/{MAX_STEPS} | "
                f"pinball={loss.item():.5f}"
            )
        step += 1
        if step >= MAX_STEPS:
            break




In [ ]:
# Save the model produced by the training cell.
checkpoint_name = (
    f"baseline-{MODEL_TAG}__freq-{FREQ}__hor-{PRIMARY_HORIZON}"
    f"__seq-{SEQ_LEN}__seed-{SEED}__steps-{MAX_STEPS}"
    f"__bs-{BATCH_SIZE}__in-{INPUT_LEN}__stride-{TRAIN_STRIDE}"
    f"__local-itran__q-10-50-90"
)
checkpoint_dir = MODEL_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / f"{checkpoint_name}.pt"
torch.save(
    {
        "model_state": model.state_dict(),
        "config": {
            "FREQ": FREQ,
            "PRIMARY_HORIZON": PRIMARY_HORIZON,
            "SEQ_LEN": SEQ_LEN,
            "INPUT_LEN": INPUT_LEN,
            "past_cov_cols": past_cov_cols,
            "future_cov_cols": future_cov_cols,
            "static_cols": static_cols,
            "hidden_size": HIDDEN_SIZE,
            "transformer_layers": TRANSFORMER_LAYERS,
            "transformer_heads": TRANSFORMER_HEADS,
        },
        "y_scaler_mean": y_scaler.mean_.tolist(),
        "y_scaler_scale": y_scaler.scale_.tolist(),
    },
    checkpoint_path,
)
print("Saved:", checkpoint_path)


In [ ]:
# Select the checkpoint to load, then run this cell instead of the training and save cells.
CHECKPOINT_TO_LOAD = (
    Path("..") / "checkpoints" / "iTransformer" /
    "baseline-itransformer__freq-30min__hor-24h__seq-48__seed-0__steps-300__bs-64__in-96__stride-48__local-itran__q-10-50-90.pt"
)

# Loads a local iTransformer checkpoint after checking its saved configuration and target scaler.
def load_checkpoint(path, *, load_optimizer=False):
    selected_path = Path(path)
    if not selected_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {selected_path}")

    checkpoint = torch.load(selected_path, map_location=DEVICE, weights_only=True)
    if "model_state" not in checkpoint:
        raise KeyError("Checkpoint does not contain model_state.")

    config = checkpoint.get("config", {})
    expected = {
        "FREQ": FREQ,
        "PRIMARY_HORIZON": PRIMARY_HORIZON,
        "SEQ_LEN": SEQ_LEN,
        "INPUT_LEN": INPUT_LEN,
        "past_cov_cols": past_cov_cols,
        "future_cov_cols": future_cov_cols,
        "static_cols": static_cols,
        "hidden_size": HIDDEN_SIZE,
        "transformer_layers": TRANSFORMER_LAYERS,
        "transformer_heads": TRANSFORMER_HEADS,
    }
    mismatches = {
        key: (config.get(key), expected_value)
        for key, expected_value in expected.items()
        if config.get(key) != expected_value
    }
    if mismatches:
        details = "; ".join(
            f"{key}: saved={saved!r}, current={current!r}"
            for key, (saved, current) in mismatches.items()
        )
        raise ValueError(f"Checkpoint settings do not match this run: {details}")

    saved_mean = np.asarray(checkpoint.get("y_scaler_mean"), dtype=np.float64)
    saved_scale = np.asarray(checkpoint.get("y_scaler_scale"), dtype=np.float64)
    if not (
        np.allclose(saved_mean, y_scaler.mean_, rtol=1e-6, atol=1e-7)
        and np.allclose(saved_scale, y_scaler.scale_, rtol=1e-6, atol=1e-7)
    ):
        raise ValueError("Checkpoint target scaling does not match the current training split.")

    state_dict = checkpoint["model_state"]
    legacy_prefixes = {
        "var_proj.": "variable_projection.",
        "future_proj.": "future_projection.",
        "static_proj.": "static_projection.",
        "head.": "forecast_head.",
    }
    translated_state = {}
    for key, value in state_dict.items():
        translated_key = key
        for old_prefix, new_prefix in legacy_prefixes.items():
            if key.startswith(old_prefix):
                translated_key = new_prefix + key[len(old_prefix):]
                break
        translated_state[translated_key] = value

    model.load_state_dict(translated_state, strict=True)
    if load_optimizer:
        raise ValueError("The iTransformer checkpoint does not store optimizer state.")
    model.to(DEVICE).eval()
    print("Loaded:", selected_path)
    return config

loaded_checkpoint_config = load_checkpoint(CHECKPOINT_TO_LOAD)


In [ ]:
def inverse_y_scaled(values):
    values = np.asarray(values, dtype=np.float32)
    log_values = y_scaler.inverse_transform(
        values.reshape(-1, 1)
    ).reshape(values.shape)
    return np.maximum(np.expm1(log_values), 0.0).astype(np.float32)


@torch.no_grad()
def evaluate_itransformer():
    raw_pool = pools["validation"]
    scaled_pool = scaled_val
    n_windows = min(EVAL_WINDOWS, scaled_pool["yf"].shape[0])

    rng = np.random.default_rng(SEED + 2000)
    indices = rng.choice(
        scaled_pool["yf"].shape[0],
        size=n_windows,
        replace=False,
    )

    quantile_batches = []
    model.eval()
    for start in range(0, n_windows, BATCH_SIZE):
        batch_indices = indices[start:start + BATCH_SIZE]
        y_past_b = torch.from_numpy(
            scaled_pool["yp"][batch_indices]
        ).to(DEVICE)
        pc_b = torch.from_numpy(
            scaled_pool["pc"][batch_indices]
        ).to(DEVICE)
        fc_b = torch.from_numpy(
            scaled_pool["fc"][batch_indices]
        ).to(DEVICE)
        xs_b = torch.from_numpy(
            scaled_pool["xs"][batch_indices]
        ).to(DEVICE)
        quantile_batches.append(
            model(y_past_b, pc_b, fc_b, xs_b).cpu().numpy()
        )

    quantiles_scaled = np.concatenate(quantile_batches, axis=0)
    y_real = raw_pool["yf"][indices].astype(np.float32)
    q10 = inverse_y_scaled(quantiles_scaled[:, :, 0:1])
    q50 = inverse_y_scaled(quantiles_scaled[:, :, 1:2])
    q90 = inverse_y_scaled(quantiles_scaled[:, :, 2:3])
    return y_real, np.minimum(q10, q90), q50, np.maximum(q10, q90)


y_real_kwh, y_q10_kwh, y_q50_kwh, y_q90_kwh = evaluate_itransformer()
metrics = compute_metrics(
    y_real_kwh,
    y_q10_kwh,
    y_q50_kwh,
    y_q90_kwh,
)
metrics_table = print_metrics_table([metrics])
